# M04 · Earthquake catalogs: read, clean, look

**Week 1 · about 8 h (+ 1.5 h optional)** · before this: M01, M02

> **한국어 요약** 이 모듈에서는 지진 목록(지진 카탈로그)에 무엇이 들어 있고 누가 만드는지(기상청, 한국지질자원연구원, ISC, 미국지질조사국 ComCat), 규모의 종류(ML, mb, Mw)가 어떻게 다른지, CSV와 QuakeML 형식이 무엇인지 배웁니다. 기상청 목록의 정밀도와 한계(0.1 단위 규모, 정수 km 깊이와 비어 있는 옛 깊이, 0.01도 단위 위치, 관측망이 늘어난 네 시기, 규모 2.0을 경계로 한 미소지진 구분, 북한과 바다의 지진)를 직접 확인합니다. 규모-빈도 분포를 계산하는 함수, 기준 시각 앞뒤의 UTC 시간 창으로 지진을 고르는 함수, 두 목록에서 같은 지진을 일대일로 짝짓는 함수를 작성하고, 시기별 완전성 규모를 그림을 보고 판단합니다. 이어서 규모 2.0 근처에서 해마다 나타나는 이상한 점을 세어 보고 b값 추정에 어떤 뜻이 있는지 설명합니다. 선택 과제에서는 ObsPy로 2019년 리지크레스트 지진 목록을 내려받아 경주 지진과 규모-빈도 분포를 비교합니다.

**You will**

- Compute non-cumulative and cumulative frequency-magnitude distributions with bins centered on multiples of the bin width, robust to floating-point noise
- Select the events of a UTC time window around a time zero written in any time zone, and explain how a foreshock or a mainshock as time zero changes the counts of a sequence
- Match the events of two catalogs one-to-one within limits on time and distance, whatever the row order
- Read the magnitude of completeness of four network epochs from FMD plots, and measure a yearly count ratio across ML 2.0 that a Gutenberg-Richter law does not explain
- Download a catalog from USGS ComCat with ObsPy, cache it, and compare its FMD with a KMA sequence (stretch)

**New words:** earthquake catalog (지진 목록, 지진 카탈로그), origin time (진원시, 발생 시각), hypocenter (진원), epicenter (진앙), focal depth (진원 깊이), magnitude (규모), magnitude type (규모 종류), local magnitude (ML; 국지 규모) · [all terms](../../../docs/glossary.md)

**Optional reading:** Mignan & Woessner (2012), the introduction and the maximum-curvature method; Gutenberg & Richter (1944), the frequency-magnitude relation (a short paper)

Log any AI help in `work/ai_log/M04.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M04.yaml](../../../.sgtl/criteria/M04.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DM = 0.1                                   # magnitude bin width: the KMA list gives ML in steps of 0.1
SEQUENCE = "gyeongju2016"                  # a named sequence from st.data.sequence (Part 4)
FORESHOCK_UTC = "2016-09-12T10:44:32Z"     # the ML 5.1 foreshock of the Gyeongju sequence (KMA, UTC)
MAINSHOCK_UTC = "2016-09-12T11:32:54Z"     # the ML 5.8 mainshock of the Gyeongju sequence (KMA, UTC)
DAYS_BEFORE = 1.0                          # time window before time zero, in days (Part 4)
DAYS_AFTER = 30.0                          # time window after time zero, in days (Part 4)
PILEUP_YEARS = range(2008, 2027)           # UTC years for task 5: 2008 up to and including 2026
USE_NETWORK = True                         # False: never contact a data center (cached copies are still used)
NETWORK_TIMEOUT_S = 60                     # give up on a data center after this many seconds
DOWNLOAD_DIR = "downloads"                 # downloaded catalogs are cached here, next to this notebook
FIGURE_DIR = "figures"                     # your figure files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M04")

import io
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Path(DOWNLOAD_DIR).mkdir(exist_ok=True)
Path(FIGURE_DIR).mkdir(exist_ok=True)

## How this notebook works

The same five steps as in M01 and M02, for every task: read the **Contract**, write a **prediction**, replace
`raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer the
**Explain** prompt in 3-5 sentences of your own. `st.hint("M04-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks of tasks 1, 2, 3 and 5 try your function on fresh made-up catalogs whose
answers are known from how they were built; `st.reroll("M04-T1")` draws new ones.

Two parts of this notebook can talk to data centers on the internet (Part 1 and the stretch task). Both are
optional: when the network is down they print a short message and the rest of the notebook works without
them. A successful download is kept in `DOWNLOAD_DIR`, so the next run reads that copy instead.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions or your Explain answers ([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in
`work/ai_log/M04.md`.

## Part 1 · What a catalog holds, and who makes it (about 50 min)

An **earthquake catalog** (glossary `earthquake_catalog`) has one row per earthquake. The core columns are:

- the **origin time** (glossary `origin_time`): when the rupture started;
- the **hypocenter** (glossary `hypocenter`): where it started, as latitude, longitude and **focal depth**
  (glossary `focal_depth`); the point on the surface above it is the **epicenter** (glossary `epicenter`);
- the **magnitude** (glossary `magnitude`), with its **magnitude type** (glossary `magnitude_type`);
- ideally the **uncertainty** of each number: how far off it could be. Location errors are often given as an
  **error ellipse** (glossary `error_ellipse`); depth and magnitude errors as plus-or-minus values.

Different agencies make different catalogs of the same earthquakes, each from its own stations and methods:

- **KMA** (Korea Meteorological Administration) runs the national network and publishes the Korean catalog that
  comes with this course (KOGL Type 1, glossary `kogl`: free reuse when the source is named).
- **KIGAM** (Korea Institute of Geoscience and Mineral Resources) runs its own stations and catalog.
- The **ISC** (glossary `isc`: International Seismological Centre) collects the reports of many agencies into
  one global bulletin; its reviewed bulletin appears about two years after the events.
- **USGS ComCat** (glossary `comcat`: the U.S. Geological Survey's Comprehensive Catalog) is global and
  updated within minutes; for Korea it lists only the larger events.

So the same earthquake has slightly different origin times, epicenters, depths and magnitudes in different
catalogs. Part 5 matches the rows of two catalogs that describe the same earthquake.

**Magnitude types.** A magnitude is a number on a logarithmic scale, but there are several ways to measure it:

- **ML**, the **local magnitude** (glossary `local_magnitude`; Richter 1935), from the largest swing on
  seismometers within a few hundred kilometers, corrected for distance. Every country tunes the correction to
  its own crust; Sheen et al. (2018) derived one for South Korea. The KMA catalog gives ML.
- **mb**, the **body-wave magnitude** (glossary `body_wave_magnitude`), from the first P waves (a period near
  one second) recorded far away. It stops growing ("saturates") for large earthquakes, near 6.
- **Mw**, the **moment magnitude** (glossary `moment_magnitude`; Hanks & Kanamori 1979), from the **seismic
  moment** (glossary `seismic_moment`), the physical size of the rupture. It does not saturate. ComCat writes
  variants such as Mww and Mwr (Mw from different kinds of waves).

For one earthquake the types can differ by several tenths: KMA gives the 2016 Gyeongju mainshock ML 5.8, and
ComCat lists it as Mww 5.4 (checked on 2026-11-15). Bormann et al. (2014) describe the international
standards. Never put magnitudes of different types into one count without saying so.

In [ ]:
cat = st.data.kma_catalog()
print(f"KMA catalog: {len(cat)} events, {cat['origin_utc'].min():%Y-%m-%d} to {cat['origin_utc'].max():%Y-%m-%d} (UTC)")
print("columns:", list(cat.columns))
print(cat.attrs["source"].splitlines()[0])
cat.loc[cat["ml"] >= 5.0, ["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]]

The KMA list has no uncertainty columns and one magnitude type. Richer catalogs need a richer format.

**CSV versus QuakeML.** A CSV file (one row per event, commas between the fields) is easy to read, but nobody
fixes its columns or units, and one row has room for one location and one magnitude only. **QuakeML**
(glossary `quakeml`; Schorlemmer et al. 2011) is the standard XML format for catalogs: every event can carry
several origins and magnitudes, each with its uncertainty, its method and the agency that made it. ObsPy
(Beyreuther et al. 2010; Krischer et al. 2015) reads and writes QuakeML as `Catalog`, `Event`, `Origin` and
`Magnitude` objects. Watch the units: **QuakeML gives depths in meters**, the KMA list in kilometers.

The cell below writes the Gyeongju mainshock (KMA values) as QuakeML in memory and reads it back:

In [ ]:
from obspy import UTCDateTime, read_events
from obspy.core.event import Catalog, Event, Magnitude, Origin

main_row = cat.loc[cat["event_id"] == "2016000291"].iloc[0]
demo_origin = Origin(time=UTCDateTime(main_row["origin_utc"].isoformat()), latitude=float(main_row["latitude"]),
                     longitude=float(main_row["longitude"]), depth=float(main_row["depth_km"]) * 1000.0)  # meters
demo_event = Event(origins=[demo_origin], magnitudes=[Magnitude(mag=float(main_row["ml"]), magnitude_type="ML")])
buffer = io.BytesIO()
Catalog(events=[demo_event]).write(buffer, format="QUAKEML")
quakeml_text = buffer.getvalue().decode("utf-8")
start = quakeml_text.find("<origin")
print(f"{len(quakeml_text)} characters of QuakeML for one event; the origin part:")
print(quakeml_text[start:quakeml_text.find("</origin>") + 9])

again = read_events(io.BytesIO(buffer.getvalue()))
o = again[0].origins[0]
print("read back:", o.time, o.latitude, o.longitude, o.depth, "m depth,", again[0].magnitudes[0].magnitude_type)

**Asking a data center.** The **FDSN** (glossary `fdsn`) defined web services that data centers answer in the
same way, so one program can ask many of them. ObsPy's FDSN client does this:
`Client("USGS").get_events(...)` asks ComCat and `Client("ISC").get_events(...)` asks the ISC; both return an
ObsPy `Catalog`. The arguments limit the request: `starttime`, `endtime` (UTC), `minlatitude`, `maxlatitude`,
`minlongitude`, `maxlongitude` (degrees; west and south are negative) and `minmagnitude`.

`events_table` below turns such a `Catalog` into a pandas table: one row per event, with the event's preferred
origin and magnitude (the ones the agency recommends), depths converted to kilometers and the uncertainties
where the agency gives them. It is a tool for this notebook, not a task.

In [ ]:
def events_table(events):
    """One row per event of an ObsPy Catalog, with its preferred (or else first) origin and magnitude.

    events  : obspy Catalog, for example from Client.get_events or read_events
    returns : pandas DataFrame with event_id (text), origin_utc (aware, UTC), latitude, longitude (degrees),
              depth_km, depth_err_km (km, NaN when not given), mag, mag_err, mag_type; sorted by origin_utc
    """
    rows = []
    for ev in events:
        org = ev.preferred_origin() or (ev.origins[0] if ev.origins else None)
        mag = ev.preferred_magnitude() or (ev.magnitudes[0] if ev.magnitudes else None)
        if org is None:
            continue
        found = re.search(r"eventid=([^&]+)", str(ev.resource_id))
        depth_m = org.depth if org.depth is not None else np.nan
        depth_err_m = org.depth_errors.uncertainty if org.depth_errors.uncertainty is not None else np.nan
        rows.append({
            "event_id": found.group(1) if found else str(ev.resource_id).rstrip("/").split("/")[-1],
            "origin_utc": pd.Timestamp(org.time.isoformat()).tz_localize("UTC"),
            "latitude": org.latitude,
            "longitude": org.longitude,
            "depth_km": depth_m / 1000.0,                 # QuakeML depths are in meters
            "depth_err_km": depth_err_m / 1000.0,
            "mag": mag.mag if mag is not None else np.nan,
            "mag_err": mag.mag_errors.uncertainty if mag is not None and mag.mag_errors.uncertainty else np.nan,
            "mag_type": mag.magnitude_type if mag is not None else None,
        })
    columns = ["event_id", "origin_utc", "latitude", "longitude", "depth_km", "depth_err_km", "mag", "mag_err",
               "mag_type"]
    table = pd.DataFrame(rows, columns=columns)
    table["origin_utc"] = pd.to_datetime(table["origin_utc"], utc=True)
    return table.sort_values("origin_utc", kind="stable").reset_index(drop=True)

The cell below asks ComCat for the events of magnitude 2.5 or more around Korea in 2016-2017. It is
**optional**: it uses a cached copy when there is one, contacts USGS only when `USE_NETWORK` is True, and when
the network fails it says so and moves on (`comcat_korea` is then `None`).

In [ ]:
comcat_cache = Path(DOWNLOAD_DIR) / "comcat_korea_2016_2017.xml"
comcat_korea = None
if comcat_cache.is_file():
    comcat_korea = events_table(read_events(str(comcat_cache)))
    print(f"Read the cached copy {comcat_cache}.")
elif USE_NETWORK:
    try:
        from obspy.clients.fdsn import Client

        korea_events = Client("USGS", timeout=NETWORK_TIMEOUT_S).get_events(
            starttime=UTCDateTime("2016-01-01"), endtime=UTCDateTime("2018-01-01"), minlatitude=33.0,
            maxlatitude=39.0, minlongitude=124.0, maxlongitude=132.0, minmagnitude=2.5)
        korea_events.write(str(comcat_cache), format="QUAKEML")
        comcat_korea = events_table(korea_events)
        print(f"Downloaded {len(korea_events)} events from ComCat; saved {comcat_cache} for the next run.")
    except Exception as exc:   # no network, a slow server, ...: this step is optional
        print(f"Could not download from ComCat ({type(exc).__name__}). This step is optional; the rest of the "
              "notebook works without it. Run this cell again later to try once more.")
else:
    print("USE_NETWORK is False and there is no cached copy: skipping the optional ComCat download.")
if comcat_korea is not None:
    print(f"{len(comcat_korea)} ComCat events; magnitude types:", comcat_korea["mag_type"].value_counts().to_dict())
    display(comcat_korea[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "depth_err_km", "mag",
                          "mag_type"]])

If the download worked, look at the table: which events are Korean, which Japanese, which magnitude types
appear, and how many depths are exactly 10 km? (A round depth such as 10 km is often a **fixed** depth: the
agency could not resolve it and set it by hand.)

## Part 2 · Precision and quirks of the KMA list (about 50 min)

Before computing anything from a catalog, find out how precise its numbers are and how its contents changed
over time. The data folder's `README.md` (`data/catalogs/`) describes the KMA file; here you check the main
points yourself.

In [ ]:
years = cat["origin_utc"].dt.year                       # UTC years, one per event
tenths = cat["ml"].to_numpy() * 10.0
print("every ML is a multiple of 0.1:", bool(np.allclose(tenths, np.round(tenths))))
print("ML from", cat["ml"].min(), "to", cat["ml"].max())
known_depth = cat["depth_km"].dropna()
print("every given depth is a whole number of km:", bool(np.allclose(known_depth, np.round(known_depth))))
print("share of blank depths, by year:")
print(cat["depth_km"].isna().groupby(years).mean().loc[2007:2016].round(2).to_string())

In [ ]:
lat10, lon10 = cat["latitude"] * 10.0, cat["longitude"] * 10.0
on_tenth = pd.Series(np.isclose(lat10, lat10.round()) & np.isclose(lon10, lon10.round()), index=cat.index)
print("share of epicenters on a 0.1-degree grid, by year:")
print(on_tenth.groupby(years).mean().loc[2001:2008].round(2).to_string())

So magnitudes come in steps of 0.1, depths in whole kilometers (blank for nearly every event before 2010 and
for most events in 2010-2011), and epicenters in steps of 0.1 degree (about 11 km) before 2005 and 0.01 degree
(about 1 km) from 2005 on. These are rounding steps, not error bars: the real errors are usually larger.

**Network epochs.** A denser network records smaller earthquakes, so what the list contains depends on how
many stations were running. We split the list into four **network epochs** (glossary `network_epoch`):
1978-2007 (few stations, almost only events of ML 2.0 or more), 2008-2015 (small events listed from 2008 on),
2016-2017 (the Gyeongju and Pohang sequences, while many new stations were being installed) and 2018-2026 (the
dense modern network).

**The line at ML 2.0.** KMA calls earthquakes smaller than ML 2.0 **microearthquakes** (glossary
`microearthquake`, 미소지진). Before 2008 the list has almost none of them.

In [ ]:
EPOCHS = {"1978-2007": (1978, 2007), "2008-2015": (2008, 2015), "2016-2017": (2016, 2017), "2018-2026": (2018, 2026)}

per_year = cat.groupby(years).size()
smallest = cat.groupby(years)["ml"].min()
fig, (ax_n, ax_m) = plt.subplots(2, 1, figsize=(7.0, 4.8), sharex=True)
ax_n.bar(per_year.index, per_year.to_numpy(), width=0.8, color="0.55")
ax_n.set_yscale("log")
ax_n.set_ylim(1, 20000)
ax_n.set_ylabel("Events per year")
ax_m.plot(smallest.index, smallest.to_numpy(), "o-", ms=3, color="tab:blue")
ax_m.set_ylabel(f"Smallest {st.plot.ML_LABEL} of the year")
ax_m.set_xlabel("Year (UTC)")
for label, (first, last) in EPOCHS.items():
    for ax in (ax_n, ax_m):
        ax.axvline(first - 0.5, color="tab:red", lw=0.8, ls="--")
    ax_n.text((first + last) / 2.0, 6000, label, ha="center", fontsize=8)
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "kma_events_per_year.png", dpi=150)

for label, (first, last) in EPOCHS.items():
    part = cat[(years >= first) & (years <= last)]
    print(f"{label}: {len(part):5d} events, smallest ML {part['ml'].min():.1f}, "
          f"{100.0 * (part['ml'] < 2.0).mean():4.1f} % below ML 2.0, {100.0 * part['depth_km'].notna().mean():5.1f} % with a depth")

**Not only South Korea.** The list also holds events in North Korea and under the seas around the peninsula.
The map shows every event as a small dot colored by epoch (PyGMT, as in M02).

In [ ]:
import pygmt

where_region = [122.5, 132.0, 32.0, 42.0]
epoch_colors = {"1978-2007": "black", "2008-2015": "dodgerblue", "2016-2017": "orange", "2018-2026": "red3"}
with st.plot.gmt_style(FONT_TITLE="12p,Helvetica"):
    fig_map = pygmt.Figure()
    fig_map.basemap(region=where_region, projection="M11c", frame=["af", "+tKMA catalog, 1978-2026"])
    fig_map.coast(land="gray92", water="white", shorelines="0.25p,gray40", borders="1/0.4p,gray50")
    for label, (first, last) in EPOCHS.items():
        part = cat[(years >= first) & (years <= last)]
        fig_map.plot(x=part["longitude"], y=part["latitude"], style="c0.05c", fill=epoch_colors[label],
                     label=f"{label}+S0.15c")
    fig_map.legend(position="jBR+o0.2c", box="+gwhite+p0.5p")
    fig_map.basemap(map_scale=st.plot.scale_bar_spec(where_region, km=200))
fig_map.show()
north = cat["latitude"] > 38.6
print(f"{north.sum()} events north of 38.6 N (North Korea and its seas); "
      f"{(cat['longitude'] > 130.0).sum()} events east of 130 E (East Sea)")

Near 41.3 N, 129.2 E (North Korea) a cluster of small events starts in September 2017; the nuclear
explosions themselves are not in this list. Events far from the stations are located less precisely and only
larger ones are recorded, so before you count or compare earthquakes, choose a region and a period on purpose.

## Part 3 · The frequency-magnitude distribution (about 70 min, then task 1)

Small earthquakes are far more common than large ones. The **frequency-magnitude distribution** (FMD, glossary
`frequency_magnitude_distribution`) counts them per magnitude:

- the **non-cumulative** FMD counts the events in each magnitude bin;
- the **cumulative** FMD (glossary `cumulative_count`) counts, for each bin, the events in that bin **or
  above**: N(≥M).

Gutenberg & Richter (1944) found that the cumulative counts fall on a straight line on a log scale, the
**Gutenberg-Richter law** (glossary `gutenberg_richter_law`): $\log_{10} N(\geq M) = a - bM$. The slope b
(glossary `b_value`) is close to 1 in most regions, so ten times fewer events per unit of magnitude.

**Bins.** The KMA list reports ML in steps of 0.1, the **magnitude bin width** Δm (glossary
`magnitude_binning`; `DM` in the parameters cell). A reported 1.6 stands for every magnitude that rounds to
1.6, from 1.55 up to 1.65. So the bins are **centered on multiples of Δm** with edges at ±Δm/2: the 1.6 bin
runs from 1.55 to 1.65, not from 1.6 to 1.7.

**Floating-point numbers.** Computers store decimal numbers in binary (glossary `floating_point_number`), so
most decimals are stored as a value very close to, but not exactly, the number you typed. Arithmetic shows
it, and a magnitude that went through a unit conversion or another program may come back as 1.5999999:

In [ ]:
print(0.1 * 3, "| equal to 0.3?", 0.1 * 3 == 0.3)
print(1.6 / 0.1, "| 1.6 / 0.1 equal to 16?", 1.6 / 0.1 == 16)
print(1.5999999 / 0.1, 1.6000001 / 0.1)
noisy = np.array([1.5999999, 1.6000001, 1.6])
print("all three stand for 1.6:", noisy, "| exactly equal to 1.6:", noisy == 1.6)

### M04-T1 · Counting events per magnitude bin · core · about 60 min

**Goal:** write `fmd(mags, dm=0.1)`: the non-cumulative and the cumulative frequency-magnitude distribution.

**Given:** magnitudes that were reported in steps of `dm`, so every value stands for a multiple of `dm`, but
some carry floating-point noise smaller than dm/1000 (1.5999999 stands for 1.6). Magnitudes can be negative
(very small events).

**Contract:**
- Inputs: `mags`, a 1-D list, numpy array or pandas Series of magnitudes (at least one value, no NaN); `dm`,
  the bin width (a positive number such as 0.1, 0.2 or 0.05; default 0.1).
- Output: a tuple `(centers, n_noncum, n_cum)` of three 1-D numpy arrays of the same length:
  - `centers`: the bin centers, multiples of `dm`, from the bin of the smallest magnitude to the bin of the
    largest, in steps of `dm`, increasing, with every bin in between (an empty bin has count 0);
  - `n_noncum`: the number of magnitudes in each bin (from center − dm/2 to center + dm/2);
  - `n_cum`: the number of magnitudes in that bin or in any higher bin (so `n_cum[0]` is the total).
- Use numpy (or pandas); a ready-made FMD function from a seismology package is not allowed.
- Example: `fmd([1.0, 1.0, 1.1, 1.3])` gives centers 1.0, 1.1, 1.2, 1.3, `n_noncum` 2, 1, 0, 1 and `n_cum`
  4, 2, 1, 1.

**Done when** `st.check("M04-T1", func=fmd)` passes on 8 fresh sets of magnitudes (bin widths 0.05, 0.1 and
0.2; noisy values; negative magnitudes; a single value; gaps). The counts are known from how the sets were
built.

**Stuck?** `st.hint("M04-T1", 1)`

In [ ]:
# Before you code: for the magnitudes 2.0, 2.0, 2.1, 2.4 (dm = 0.1), what do you expect for the three arrays?
st.predict("M04-T1", """

""")

In [ ]:
def fmd(mags, dm=0.1):
    """Non-cumulative and cumulative frequency-magnitude distribution.

    mags    : 1-D list, numpy array or pandas Series of magnitudes reported in steps of dm (may carry tiny
              floating-point noise, may be negative)
    dm      : float, bin width (> 0); bins are centered on multiples of dm, with edges at center +- dm/2
    returns : (centers, n_noncum, n_cum), three 1-D numpy arrays of the same length: bin centers from the
              smallest to the largest bin (empty bins included), the count in each bin, and the count in each
              bin or above
    """
    # Step 1: give every magnitude its bin number: the whole number k whose bin (k*dm - dm/2 to k*dm + dm/2)
    #         holds it. Think about what floating-point noise does here.
    # Step 2: count the magnitudes in every bin from the smallest k to the largest, empty bins included.
    # Step 3: the cumulative count of a bin adds its own count to the counts of all higher bins.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T1: write your code here")

In [ ]:
st.check("M04-T1", func=fmd)

**Explain (3-5 sentences):** Why are the bins centered on multiples of dm, with edges at ±dm/2, instead of
running from one multiple to the next? What would `np.floor(mags / dm)` do to a value stored as 1.5999999?
Why does the cumulative count never rise from one bin to the next, and why do FMD plots use a log scale?

Your function on the whole KMA list:

In [ ]:
centers_all, n_all, cum_all = fmd(cat["ml"], DM)
fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.semilogy(centers_all, cum_all, "s", ms=3, color="0.35", label="Cumulative, N(≥M)")
ax.semilogy(centers_all, np.where(n_all > 0, n_all, np.nan), "o", ms=3.5, color="tab:blue", label="Per bin")
ax.set_xlabel(f"Local magnitude {st.plot.ML_LABEL}")
ax.set_ylabel("Number of events")
ax.set_title("KMA catalog, 1978-2026")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "kma_fmd_all.png", dpi=150)
print(f"{len(centers_all)} bins from {centers_all[0]:.1f} to {centers_all[-1]:.1f}; {cum_all[0]} events in all; "
      f"largest bin count {n_all.max()} at ML {centers_all[np.argmax(n_all)]:.1f}")

## Part 4 · Sequences and time zero (about 60 min, then task 2)

An **earthquake sequence** (glossary `earthquake_sequence`) is a group of earthquakes close in space and time.
Its largest event is the **mainshock** (glossary `mainshock`); earlier events nearby are **foreshocks**
(glossary `foreshock`) and later ones **aftershocks** (glossary `aftershock`). The 2016 Gyeongju sequence
(Woo et al. 2019) started with an ML 5.1 foreshock at 10:44:32 UTC on 12 September (19:44:32 KST), and the
ML 5.8 mainshock followed 48 minutes later at 11:32:54 UTC (20:32:54 KST).

Two plots show a sequence: a **time-magnitude plot** (glossary `time_magnitude_plot`, as in M02) and the
**cumulative number** of events against time (glossary `cumulative_count`), which climbs steeply right after
a large event and flattens as the aftershocks die down. Times are counted from a **time zero** (glossary
`time_zero`), the moment chosen as t = 0, usually the mainshock origin time. Here both are drawn with the
mainshock as time zero; the right panels zoom in on the first hours.

In [ ]:
seq = st.data.sequence(SEQUENCE)
print(seq.attrs["sequence"]["description"], "-", len(seq), "events within",
      seq.attrs["sequence"]["radius_km"], "km of the center")
print(seq.nlargest(3, "ml")[["event_id", "origin_utc", "origin_kst", "ml"]].to_string(index=False))

t_main = pd.Timestamp(MAINSHOCK_UTC)
t_fore = pd.Timestamp(FORESHOCK_UTC)
since_days = ((seq["origin_utc"] - t_main) / pd.Timedelta(days=1)).to_numpy()
since_hours = since_days * 24.0
running = np.arange(1, len(seq) + 1)
fig, axes = plt.subplots(2, 2, figsize=(8.5, 5.2), sharex="col")
axes[0, 0].plot(since_days, seq["ml"], "o", ms=2.5, mfc="none", mec="tab:blue", mew=0.5)
axes[1, 0].step(np.sort(since_days), running, where="post", color="tab:blue")
early = (since_hours >= -2.0) & (since_hours <= 6.0)
axes[0, 1].plot(since_hours[early], seq["ml"].to_numpy()[early], "o", ms=3, mfc="none", mec="tab:blue", mew=0.6)
axes[1, 1].step(np.sort(since_hours[early]), np.arange(1, early.sum() + 1), where="post", color="tab:blue")
for ax in axes[:, 1]:
    ax.axvline((t_fore - t_main) / pd.Timedelta(hours=1), color="tab:red", lw=0.8, ls="--")
axes[0, 0].set_ylabel(st.plot.ML_LABEL)
axes[1, 0].set_ylabel("Cumulative number")
axes[1, 0].set_xlabel("Days since the mainshock")
axes[1, 1].set_xlabel("Hours since the mainshock (dashed: foreshock)")
axes[0, 0].set_title("Whole sequence")
axes[0, 1].set_title("First hours")
fig.tight_layout()
fig.savefig(Path(FIGURE_DIR) / "gyeongju_time_magnitude_cumulative.png", dpi=150)
print(f"{early.sum()} events from 2 h before to 6 h after the mainshock; "
      f"foreshock at {(t_fore - t_main) / pd.Timedelta(minutes=1):.1f} min")

### M04-T2 · Events around a chosen moment · core · about 60 min

**Goal:** write `select_window(df, t0_utc, days_before, days_after)`: the events of a catalog from
`days_before` days before a time zero to `days_after` days after it.

**Given:** a catalog DataFrame with at least `origin_utc` (time-zone-aware, UTC), like `st.data.sequence(...)`.
Times of earthquakes reach you in many spellings: `"2016-09-12T11:32:54Z"`, `"2016-09-12T20:32:54+09:00"`
(the same moment in KST), a pandas `Timestamp`, or a time without a zone.

**Contract:**
- Inputs: `df`; `t0_utc`, the time zero: text or a pandas `Timestamp` (or `datetime`). With a time zone or an
  offset, it is that moment; without one, it is read as UTC. `days_before`, `days_after`: numbers of days
  (0 or more, fractions allowed).
- Output: a DataFrame with the rows of `df` whose origin time lies from `t0 − days_before` to
  `t0 + days_after`, **both ends included**, with all of `df`'s columns, sorted by `origin_utc` from earliest
  to latest. When no event is inside: an empty DataFrame with the same columns.
- Do not change `df`.

**Done when** `st.check("M04-T2", func=select_window)` passes on 8 fresh made-up catalogs, with time zeros in
all the spellings above and events placed exactly on the ends of the windows.

**Stuck?** `st.hint("M04-T2", 1)`

In [ ]:
# Before you code: with the mainshock as time zero and DAYS_BEFORE = 1, is the foreshock inside the window?
# About how many events do you expect in the first day after the mainshock?
st.predict("M04-T2", """

""")

In [ ]:
def select_window(df, t0_utc, days_before, days_after):
    """Events from days_before days before a time zero to days_after days after it, earliest first.

    df          : pandas DataFrame with at least origin_utc (time-zone-aware, UTC)
    t0_utc      : time zero, as text or a pandas Timestamp/datetime; with a zone or offset it is that moment,
                  without one it is UTC
    days_before : float, days before time zero (>= 0)
    days_after  : float, days after time zero (>= 0)
    returns     : pandas DataFrame, the rows of df from t0 - days_before to t0 + days_after (both ends
                  included), all columns, sorted by origin_utc; df itself is not changed
    """
    # Step 1: turn t0_utc into one time-zone-aware moment in UTC, whatever spelling it came in.
    # Step 2: the two ends of the window, then keep the rows between them (ends included) and sort them.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T2: write your code here")

In [ ]:
st.check("M04-T2", func=select_window)

In [ ]:
counts = []
for name, zero_text in (("foreshock", FORESHOCK_UTC), ("mainshock", MAINSHOCK_UTC)):
    zero = pd.Timestamp(zero_text)
    window = select_window(seq, zero_text, DAYS_BEFORE, DAYS_AFTER)
    after = window["origin_utc"] > zero
    counts.append({"time zero": name, "events in window": len(window),
                   "before time zero": int((window["origin_utc"] < zero).sum()),
                   "after time zero": int(after.sum()),
                   "first hour after": int((after & (window["origin_utc"] <= zero + pd.Timedelta(hours=1))).sum()),
                   "first day after": int((after & (window["origin_utc"] <= zero + pd.Timedelta(days=1))).sum())})
pd.DataFrame(counts)

**Explain (3-5 sentences):** Look at the table above. How many events lie between the foreshock and the
mainshock, and on which side of time zero do they land for each choice? Which time zero would you use to count
the aftershocks of the Gyeongju sequence, and why? Your function got time zeros in UTC, in KST and without a
zone: what had to happen to each before the comparison with `origin_utc`?

## Part 5 · The same earthquake in two catalogs (about 30 min, then task 3)

**Catalog matching** (glossary `catalog_matching`) pairs the rows of two catalogs that describe the same
earthquake. Two events are **candidates** when their origin times differ by at most `max_dt_s` seconds and
their epicenters by at most `max_dist_km` kilometers (a great-circle distance, glossary
`great_circle_distance`). A real earthquake appears once in each catalog, so the matching must be
**one-to-one**: one event of catalog A pairs with at most one event of catalog B, and the other way round.
When an event has several candidates, the pair with the **smaller time difference** wins, because origin times
are usually far more precise than epicenters.

`epicentral_km` gives great-circle distances in km on a sphere of radius 6371 km (from ObsPy), for single
numbers or whole arrays. Use it in task 3 (or your own `haversine_km` from M01).

In [ ]:
from obspy.geodetics import locations2degrees

KM_PER_DEGREE = 6371.0 * np.pi / 180.0


def epicentral_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km (sphere of radius 6371 km) between points given in degrees."""
    arc = locations2degrees(np.asarray(lat1, dtype=float), np.asarray(lon1, dtype=float),
                            np.asarray(lat2, dtype=float), np.asarray(lon2, dtype=float))
    return np.asarray(arc, dtype=float) * KM_PER_DEGREE


print(f"Gyeongju to Pohang mainshock: {float(epicentral_km(35.76, 129.19, 36.11, 129.37)):.1f} km")

### M04-T3 · Pairing two catalogs · core · about 120 min

**Goal:** write `match_events(cat_a, cat_b, max_dt_s, max_dist_km)`: the one-to-one pairs of events that two
catalogs share.

**Given:** two catalog DataFrames with at least `event_id` (text), `origin_utc` (time-zone-aware, UTC),
`latitude` and `longitude` (degrees). Their rows can be in any order. The function `epicentral_km` above.

**Contract:**
- Inputs: `cat_a`, `cat_b`; `max_dt_s` (seconds) and `max_dist_km` (km), both positive.
- Candidates: an event a of `cat_a` and an event b of `cat_b` with |t_b − t_a| ≤ `max_dt_s` and epicentral
  distance ≤ `max_dist_km`. Event b may come before or after event a.
- One-to-one rule: take the candidate pairs in order of increasing |t_b − t_a| (smallest first) and keep a
  pair only when neither of its two events is already in a kept pair. (The check never has two candidate
  pairs with the same time difference.)
- Output: a DataFrame with the columns `id_a`, `id_b`, `dt_s` and `dist_km`, one row per kept pair, rows in
  any order. `dt_s` = t_b − t_a in seconds (negative when b came first); `dist_km` = the epicentral distance.
  No pair: an empty DataFrame with these four columns.
- Do not change `cat_a` or `cat_b`.

**Done when** `st.check("M04-T3", func=match_events)` passes on 6 fresh pairs of made-up catalogs. They hold
true pairs (b earlier or later), events found by one catalog only, decoys that are close in time but too far
away (or close but too late), and competitors: second candidates that are farther in time than the true
partner. No event lies near a limit, so any reasonable distance formula gives the same pairs.

**Stuck?** `st.hint("M04-T3", 1)`

In [ ]:
# Before you code: KMA and USGS locate the Gyeongju mainshock with different stations. How far apart do you
# expect their origin times (seconds) and epicenters (kilometers) to be?
st.predict("M04-T3", """

""")

In [ ]:
def match_events(cat_a, cat_b, max_dt_s, max_dist_km):
    """One-to-one pairs of events shared by two catalogs, closest in time first.

    cat_a, cat_b : pandas DataFrames with event_id, origin_utc (aware, UTC), latitude, longitude (degrees);
                   rows in any order
    max_dt_s     : float, largest allowed |t_b - t_a| in seconds
    max_dist_km  : float, largest allowed epicentral distance in km
    returns      : pandas DataFrame with columns id_a, id_b, dt_s (t_b - t_a, seconds), dist_km; one row per
                   kept pair; cat_a and cat_b are not changed
    """
    # Step 1: every candidate pair: all combinations of a and b, with their time difference and distance,
    #         kept when both are within the limits.
    # Step 2: go through the candidates from the smallest |time difference| up, and keep a pair only when
    #         neither event has a partner yet.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T3: write your code here")

In [ ]:
st.check("M04-T3", func=match_events)

In [ ]:
if comcat_korea is None:
    print("No ComCat table (no network and no cached copy), so there is nothing to compare. This step is optional.")
else:
    kma_1617 = cat[(cat["origin_utc"] >= pd.Timestamp("2016-01-01", tz="UTC"))
                   & (cat["origin_utc"] < pd.Timestamp("2018-01-01", tz="UTC"))]
    shared = match_events(kma_1617, comcat_korea, 10.0, 30.0)
    shared = shared.merge(kma_1617[["event_id", "ml"]], left_on="id_a", right_on="event_id").drop(columns="event_id")
    shared = shared.merge(comcat_korea[["event_id", "mag", "mag_type", "depth_km"]], left_on="id_b",
                          right_on="event_id").drop(columns="event_id")
    print(f"{len(shared)} of {len(comcat_korea)} ComCat events have a KMA partner "
          f"(KMA lists {len(kma_1617)} events in 2016-2017)")
    display(shared.round(2))

**Explain (3-5 sentences):** If the ComCat download worked, describe the table above: how large are the time
and distance differences of the pairs, and how do the KMA ML values compare with ComCat's magnitudes and types?
(Without the download, describe what you expect, using Part 1.) Why must the matching be one-to-one, and why
should the smaller time difference decide between two candidates?

## Part 6 · Completeness changes with the network (about 40 min, then task 4)

A catalog misses some of the small events: their waves are too weak at too few stations. The **magnitude of
completeness** Mc (glossary `magnitude_of_completeness`) is the smallest magnitude above which the catalog
records (almost) every earthquake in a region and period (Mignan & Woessner 2012). Above Mc the
non-cumulative FMD falls steadily as magnitude rises; below Mc it falls off, because events are missing.

A quick way to read Mc from a plot is **maximum curvature** (glossary `maximum_curvature`; Wiemer & Wyss
2000): find the bin where the non-cumulative count is highest (the peak). The peak alone tends to come out too
low, so a common rule of thumb adds 0.2 (Woessner & Wiemer 2005). Mc changes when the network changes, so read
it for each network epoch separately. The figure uses your `fmd` from task 1.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.5, 6.0), sharex=True)
for ax, (label, (first, last)) in zip(axes.ravel(), EPOCHS.items()):
    part = cat[(years >= first) & (years <= last)]
    centers_e, n_e, cum_e = fmd(part["ml"], DM)
    ax.semilogy(centers_e, cum_e, "s", ms=2.5, color="0.4", label="Cumulative, N(≥M)")
    ax.semilogy(centers_e, np.where(n_e > 0, n_e, np.nan), "o", ms=3.5, color="tab:blue", label="Per bin")
    ax.set_title(f"{label}: {len(part)} events")
    ax.grid(True, which="major", lw=0.3, color="0.85")
    ax.set_xticks(np.arange(0.0, 6.5, 0.5), minor=True)
for ax in axes[1]:
    ax.set_xlabel(f"Local magnitude {st.plot.ML_LABEL} (bins of {DM:g})")
for ax in axes[:, 0]:
    ax.set_ylabel("Number of events")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(Path(FIGURE_DIR) / "kma_fmd_by_epoch.png", dpi=150)

### M04-T4 · Completeness by epoch · core · about 40 min

**Goal:** read the magnitude of completeness of each network epoch from the FMD plots above.

**Given:** the four FMD panels; the rule of thumb of Part 6 (the peak of the non-cumulative FMD, plus 0.2).
Zoom in with `plt.xlim` in a new cell if you need to.

**Contract:**
- Write `mc_by_eye`, a dict with exactly the keys `"1978-2007"`, `"2008-2015"`, `"2016-2017"` and
  `"2018-2026"` (the keys of `EPOCHS`), each with your Mc for that epoch: a number (ML, one decimal).
- Read them from the plots. You do not need code for this task.
- Hand the check your values together with your `fmd` of task 1, which drew the panels.

**Done when** `st.check("M04-T4", answer={"mc_by_eye": mc_by_eye, "fmd": fmd})` passes: the check runs your
`fmd` on a few made-up catalogs of task 1 again, then on each epoch of the shipped catalog, and every value of
yours must be within 0.3 of the tallest per-bin count of your own FMD plus 0.2. It counts nothing itself and
prints no values.

**Stuck?** `st.hint("M04-T4", 1)`

In [ ]:
# Before you code: which epoch do you expect to have the highest Mc, and which the lowest? Why?
st.predict("M04-T4", """

""")

In [ ]:
# Shape of the answer: mc_by_eye = {"1978-2007": <Mc>, "2008-2015": <Mc>, "2016-2017": <Mc>, "2018-2026": <Mc>}
# YOUR CODE HERE
raise NotImplementedError("M04-T4: write your code here")

In [ ]:
st.check("M04-T4", answer={"mc_by_eye": mc_by_eye, "fmd": fmd})

**Explain (3-5 sentences):** Which epoch has the highest Mc and which the lowest, and how does that follow
from the network? Does every FMD fall steadily above your Mc? Look closely at 2008-2015 and 2018-2026 near
ML 2.0: describe what you see there.

## Part 7 · Something odd near one magnitude (about 45 min, then task 5)

Look again at the 2018-2026 panel above, near ML 2.0. To measure what you see there year by year, compare the
number of events in two pairs of bins just below and just above ML 2.0.

### M04-T5 · Something odd near one magnitude · core · about 60 min

**Goal:** write `pileup_ratio(df, years, low=(1.8, 1.9), high=(2.0, 2.1))`: for each year, the number of
events in the `low` bins divided by the number in the `high` bins.

**Given:** a catalog DataFrame with at least `origin_utc` (time-zone-aware, UTC) and `ml`, with magnitudes in
steps of 0.1 that may carry floating-point noise (1.8999999 stands for 1.9).

**Contract:**
- Inputs: `df`; `years`, the whole-number years to report (a list, range or array), counted in **UTC**;
  `low` and `high`, tuples of magnitude bins (multiples of 0.1). An event counts for a bin when its ML lies
  within ±0.05 of that bin.
- Output: a pandas Series indexed by `years` (in the order given) with N(low) / N(high) for each year, as
  floats. When a year has no event in the `high` bins, its value is NaN (not infinity, not 0); a year with
  events in `high` but none in `low` gives 0.0.
- Do not change `df`.

**Done when** `st.check("M04-T5", func=pileup_ratio)` passes on 7 fresh made-up catalogs (Gutenberg-Richter
magnitudes with something odd added in some years, events close to New Year, noisy magnitudes, other bins) and
two property tests (the order of the rows and listing every event twice must not change the ratios).

**Stuck?** `st.hint("M04-T5", 1)`

In [ ]:
# Before you code: what value do you expect for the ratio in a typical recent year of the KMA list?
st.predict("M04-T5", """

""")

In [ ]:
def pileup_ratio(df, years, low=(1.8, 1.9), high=(2.0, 2.1)):
    """Per UTC year: the number of events in the low bins divided by the number in the high bins.

    df      : pandas DataFrame with origin_utc (aware, UTC) and ml (steps of 0.1, may carry tiny noise)
    years   : whole-number years to report (list, range or array), in UTC
    low     : tuple of magnitude bins (multiples of 0.1) counted for the numerator
    high    : tuple of magnitude bins (multiples of 0.1) counted for the denominator
    returns : pandas Series indexed by years (in the order given): N(low) / N(high), NaN when N(high) is 0;
              df itself is not changed
    """
    # Step 1: put every magnitude into its 0.1 bin, robust to floating-point noise.
    # Step 2: count, for each year, the events in the low bins and in the high bins (years in UTC).
    # Step 3: divide, with NaN where the high count is zero, and keep the years in the order given.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T5: write your code here")

In [ ]:
st.check("M04-T5", func=pileup_ratio)

In [ ]:
kma_ratio = pileup_ratio(cat, PILEUP_YEARS)
practice = st.synth.synthetic_catalog(np.random.default_rng(2026), 300_000, start="2008-01-01", end="2027-01-01",
                                      b=1.0, mc=0.5)   # made-up Gutenberg-Richter catalog, b = 1, no oddity
practice_ratio = pileup_ratio(practice, PILEUP_YEARS)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(kma_ratio.index, kma_ratio.to_numpy(), "o-", ms=4, color="tab:red", label="KMA catalog")
ax.plot(practice_ratio.index, practice_ratio.to_numpy(), "s--", ms=3, color="0.5",
        label="Made-up catalog, b = 1")
ax.set_xlabel("Year (UTC)")
ax.set_ylabel("N(ML 1.8-1.9) / N(ML 2.0-2.1)")
fig.legend(loc="upper center", ncol=2, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "kma_ratio_by_year.png", dpi=150)
pd.DataFrame({"KMA": kma_ratio.round(2), "made-up, b = 1": practice_ratio.round(2)})

**Explain (3-5 sentences):** What does a Gutenberg-Richter law with b = 1 predict for N(1.8-1.9) / N(2.0-2.1)?
(Work it out from $\log_{10} N(\geq M) = a - bM$, with bin edges at ±0.05.) How does the KMA ratio compare,
and in which years? What would happen if you fitted one b-value to magnitudes on both sides of ML 2.0, and
which Mc would you choose for a b-value of the 2018-2026 list?

## Part 8 · Stretch: Ridgecrest 2019 from ComCat (about 90 min, optional)

The 2019 Ridgecrest sequence in California started with an Mw 6.4 foreshock on 4 July and an Mw 7.1 mainshock
on 6 July (Ross et al. 2019). Its catalog comes from a dense network with magnitudes in steps of 0.01 and of
several types. Comparing its FMD with Gyeongju's shows what changes from one catalog to another.

### M04-T6 · Ridgecrest and Gyeongju side by side · stretch · about 90 min

**Goal:** download the 2019 Ridgecrest events from USGS ComCat with ObsPy, keep a cached copy, and compare
their FMD with that of the Gyeongju sequence in one figure.

**Given:** ObsPy's FDSN client (Part 1), `events_table`, your `fmd` and `st.data.sequence("gyeongju2016")`.
The request: 35.4 to 36.1 N, 117.9 to 117.2 W, from 2019-07-04 00:00 to 2019-08-04 00:00 UTC, magnitude 2.5 or
more.

**Contract:**
- Write `ridgecrest_events(cache_csv)`: if the file `cache_csv` exists, read it (with `origin_utc` as aware
  UTC times); otherwise ask ComCat (`Client("USGS")`, timeout `NETWORK_TIMEOUT_S`, only when `USE_NETWORK` is
  True), turn the answer into a table with `events_table`, save it to `cache_csv` and return it. When the
  download fails, print one short sentence and return `None`; never stop the notebook.
- Then draw both FMDs (counts per bin and cumulative, bin width `DM`) in one figure in the house style, save it
  as a PNG in `FIGURE_DIR`, and print the number of events of each catalog and the magnitude types in the
  Ridgecrest table.
- Finally set `ridgecrest_files = {"csv": <path of the cache>, "figure": <path of the PNG>}`.

**Done when** `st.check("M04-T6", answer=ridgecrest_files)` passes. The check reads only your two files (no
network): the table must hold at least 1000 events, all inside the requested box, time span and magnitude
range, with depths in kilometers; the figure must exist and be at least as new as the table.

**Stuck?** `st.hint("M04-T6", 1)`

In [ ]:
# Before you code: which sequence do you expect to have the steeper FMD (larger b-value), and about how many
# Ridgecrest events of magnitude 2.5 or more do you expect in the first month?
st.predict("M04-T6", """

""")

In [ ]:
def ridgecrest_events(cache_csv):
    """Ridgecrest 2019 events (35.4-36.1 N, 117.9-117.2 W, 2019-07-04 to 2019-08-04 UTC, magnitude >= 2.5).

    cache_csv : str or Path of the CSV cache: read when it exists, written after a successful download
    returns   : pandas DataFrame like events_table() gives (origin_utc aware, UTC), or None when there is no
                cache and the download is not possible
    """
    # Step 1: the cached copy, if there is one.
    # Step 2: otherwise the request to ComCat (guarded: a failure prints a sentence and returns None).
    # Step 3: save the new table, so the next run does not need the network.
    # YOUR CODE HERE
    raise NotImplementedError("M04-T6: write your code here")

In [ ]:
# Call ridgecrest_events with a cache path in DOWNLOAD_DIR; draw both FMDs in one figure (Gyeongju column "ml",
# Ridgecrest column "mag"); save it as a PNG in FIGURE_DIR; print the counts and the Ridgecrest magnitude types;
# then set ridgecrest_files. Keep the figure working when the Ridgecrest table is None (no network).
# YOUR CODE HERE
raise NotImplementedError("M04-T6: write your code here")

In [ ]:
st.check("M04-T6", answer=ridgecrest_files)

**Explain (3-5 sentences):** Compare the two FMDs: the numbers of events, the magnitude steps (0.1 for KMA,
0.01 for ComCat), the magnitude types in the Ridgecrest table, and the slopes. Why is the first Ridgecrest bin
(2.5) lower than the next one? Can you compare the two b-values directly? Why or why not?

## Wrap-up

Your progress in this module (from `work/.sgtl/progress.jsonl`):

In [ ]:
st.progress("M04")

### Going further (optional, not checked)

1. Set `SEQUENCE = "pohang2017"` and the two times to the Pohang mainshock (2017-11-15T05:29:31Z) in the
   parameters cell and run Part 4 again. How do the first day and the first month compare with Gyeongju?
   (Kim et al. 2018 and Grigoli et al. 2018 discuss whether Pohang was induced.)
2. With `fmd`, draw the FMD of the 2018-2026 events north of 38.6 N and of those in South Korea. Which has the
   higher Mc, and why?
3. Ask the ISC for the same 2016-2017 Korean events with `Client("ISC").get_events(...)` (it can be slow) and
   match them with KMA and ComCat. Which magnitude types does the ISC list?

Next: M05 turns these catalogs into statistics: b-values with uncertainties, Omori-Utsu decay and declustering.